In [9]:
import pandas as pd
import numpy as np
import random
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, accuracy_score

In [10]:
feature_cols = [
    "input_count",
    "is_post",
    "is_login",
    "has_csrf",
    "suspicious_sql_fields",
    "suspicious_xss_fields",
    "security_score",
    "missing_secure_cookie",
    "missing_httponly_cookie",
    "scripts_count",
    "api_exposure"
]

label_cols = [
    "sql_injection",
    "xss",
    "csrf",
    "security_misconfiguration",
    "insecure_cookie"
]

In [11]:
def generate_sample():
    input_count = random.randint(1, 8)
    is_post = random.choice([0, 1])
    is_login = random.choice([0, 1])
    has_csrf = random.choice([0, 1])
    suspicious_sql_fields = random.randint(0, 4)
    suspicious_xss_fields = random.randint(0, 4)
    security_score = random.randint(20, 100)
    missing_secure_cookie = random.choice([0, 1])
    missing_httponly_cookie = random.choice([0, 1])
    scripts_count = random.randint(0, 15)
    api_exposure = random.choice([0, 1])

    sql_signal = suspicious_sql_fields*0.9 + (1-is_post)*0.5 + (input_count/8) + random.uniform(-1.2,1.2)
    xss_signal = suspicious_xss_fields*0.9 + (scripts_count/6) + random.uniform(-1.3,1.3)
    csrf_signal = is_post*1.2 + (1-has_csrf)*1.4 + random.uniform(-1.0,1.0)
    sec_signal = (65-security_score)/10 + api_exposure*0.4 + random.uniform(-1.0,1.0)
    cookie_signal = missing_secure_cookie + missing_httponly_cookie + random.uniform(-0.8,0.8)

    sql_label = 1 if sql_signal > 1.8 else 0
    xss_label = 1 if xss_signal > 1.9 else 0
    csrf_label = 1 if csrf_signal > 1.4 else 0
    sec_label = 1 if sec_signal > 0.6 else 0
    cookie_label = 1 if cookie_signal > 0.7 else 0

    return [
        input_count,
        is_post,
        is_login,
        has_csrf,
        suspicious_sql_fields,
        suspicious_xss_fields,
        security_score,
        missing_secure_cookie,
        missing_httponly_cookie,
        scripts_count,
        api_exposure
    ], [
        sql_label,
        xss_label,
        csrf_label,
        sec_label,
        cookie_label
    ]

In [12]:
X = []
Y = []

for _ in range(6000):
    features, labels = generate_sample()
    X.append(features)
    Y.append(labels)

X = pd.DataFrame(X, columns=feature_cols)
Y = pd.DataFrame(Y, columns=label_cols)

print("Feature Dataset Shape:", X.shape)
print("Label Dataset Shape:", Y.shape)

Feature Dataset Shape: (6000, 11)
Label Dataset Shape: (6000, 5)


In [13]:
display(X.head())
display(Y.head())

,input_count,is_post,is_login,has_csrf,suspicious_sql_fields,suspicious_xss_fields,security_score,missing_secure_cookie,missing_httponly_cookie,scripts_count,api_exposure
0,5,0,0,0,1,0,46,0,1,12,1
1,4,1,1,0,3,0,57,1,0,0,1
2,2,1,1,0,2,4,88,1,1,1,0
3,3,0,0,1,1,2,71,1,1,0,1
4,1,1,0,1,0,2,73,1,0,5,1


,sql_injection,xss,csrf,security_misconfiguration,insecure_cookie
0,1,1,1,1,0
1,1,0,1,1,0
2,0,1,1,0,1
3,1,1,0,0,1
4,0,1,0,0,0


In [14]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

Training Samples: 4800
Testing Samples: 1200


In [15]:
base_model = RandomForestClassifier(
    n_estimators=180,
    max_depth=10,
    random_state=42
)

model = MultiOutputClassifier(base_model)
model.fit(X_train, Y_train)

print("Training Complete")

Training Complete


In [16]:
preds = model.predict(X_test)

print("Overall Exact Match Accuracy:", accuracy_score(Y_test, preds))

for i, label in enumerate(label_cols):
    print(f"\n--- {label.upper()} ---")
    print(classification_report(Y_test.iloc[:, i], preds[:, i]))

Overall Exact Match Accuracy: 0.46166666666666667

--- SQL_INJECTION ---
              precision    recall  f1-score   support

           0       0.79      0.80      0.79       370
           1       0.91      0.90      0.91       830

    accuracy                           0.87      1200
   macro avg       0.85      0.85      0.85      1200
weighted avg       0.87      0.87      0.87      1200


--- XSS ---
              precision    recall  f1-score   support

           0       0.80      0.75      0.77       318
           1       0.91      0.93      0.92       882

    accuracy                           0.88      1200
   macro avg       0.85      0.84      0.85      1200
weighted avg       0.88      0.88      0.88      1200


--- CSRF ---
              precision    recall  f1-score   support

           0       0.76      0.82      0.79       624
           1       0.78      0.71      0.75       576

    accuracy                           0.77      1200
   macro avg       0.77     

In [17]:
joblib.dump(model, "web_vuln_ml_model.pkl")
print("Model saved successfully")

Model saved successfully
